# 🌍 Data Collection: Thu thập Dữ liệu Chất lượng Không khí & Thời tiết Việt Nam

Thu thập dữ liệu **lịch sử theo giờ (hourly)** từ Open-Meteo API cho 63 tỉnh thành.  
Kỹ thuật sử dụng:
1. **OpenStreetMap Nominatim**: Geocoding tọa độ GPS miễn phí, không cần API key.
2. **Open-Meteo Air Quality API**: Thu thập 8 chỉ số không khí hourly, tự động chia chunk 6 tháng.

---
## 📑 Mục lục
1. [Thiết lập Môi trường](#1)
2. [Crawl Tọa độ Địa lý](#2)
3. [Kiểm tra API (Debug)](#debug)
4. [Crawl Dữ liệu Không khí Lịch sử (Hourly)](#3)
5. [Kiểm thử Dữ liệu](#4)

<a id="1"></a>
## ⚙️ 1. Thiết lập Môi trường & Thư viện

In [34]:
import os
import time
import json
import requests
import pandas as pd
from datetime import date, datetime, timedelta
from IPython.display import display

START_DATE  = "2024-01-01"
END_DATE    = date.today().isoformat()
HOURLY_VARS = "pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,dust,european_aqi"

n_days   = (datetime.strptime(END_DATE, '%Y-%m-%d') - datetime.strptime(START_DATE, '%Y-%m-%d')).days
est_rows = n_days * 24 * 63
print("✅ Đã thiết lập môi trường thành công!")
print(f"   Khoảng thời gian : {START_DATE} → {END_DATE}")
print(f"   Ước tính số dòng : ~{est_rows:,}")

✅ Đã thiết lập môi trường thành công!
   Khoảng thời gian : 2024-01-01 → 2026-07-06
   Ước tính số dòng : ~1,386,504


<a id="2"></a>
## 📍 2. Crawl Tọa độ Địa lý (Geocoding)

In [35]:
MANUAL_COORDS = {
    'Hải Dương':          (20.9333, 106.3333),
    'Sóc Trăng':          (9.6033,  105.9800),
    'Bà Rịa - Vũng Tàu': (10.4963, 107.1688),
    'Bình Thuận':         (11.0905, 108.0721)
}

def get_osm_coordinates(province_name, session=None, timeout=15):
    if province_name in MANUAL_COORDS:
        return MANUAL_COORDS[province_name]
    url     = "https://nominatim.openstreetmap.org/search"
    params  = {"q": f"{province_name.strip()}, Vietnam", "format": "json", "limit": 1}
    headers = {"User-Agent": "AirQualityVN_StudentProject/1.0"}
    sess    = session or requests
    try:
        r = sess.get(url, params=params, headers=headers, timeout=timeout)
        r.raise_for_status()
        data = r.json()
        if data:
            return round(float(data[0]["lat"]), 4), round(float(data[0]["lon"]), 4)
    except Exception:
        pass
    return None, None

print("✅ Đã định nghĩa hàm geocode.")

✅ Đã định nghĩa hàm geocode.


In [36]:
with open('../data/provinces_list.json', 'r', encoding='utf-8') as f:
    PROVINCES_LIST = json.load(f)

PROVINCES_JSON_PATH = '../data/provinces.json'
need_geocode = True

if os.path.exists(PROVINCES_JSON_PATH):
    with open(PROVINCES_JSON_PATH, 'r', encoding='utf-8') as f:
        existing = json.load(f)
    valid_count = sum(1 for p in existing if p.get('Latitude') is not None)
    if valid_count >= len(PROVINCES_LIST):
        need_geocode = False
        print(f"✅ provinces.json đã có đủ {valid_count} tỉnh → Bỏ qua Geocoding.")
        coords_data = existing

if need_geocode:
    print("Bắt đầu lấy tọa độ bằng OpenStreetMap...")
    print("-" * 60)
    coords_data = []
    with requests.Session() as session:
        for i, p in enumerate(PROVINCES_LIST, 1):
            name = p['name']
            print(f"[{i:02d}/{len(PROVINCES_LIST)}] {name:<25}", end="")
            lat, lon = get_osm_coordinates(name, session=session)
            coords_data.append({"Code": p['code'], "Province": name,
                                 "Region": p.get('region',''), "Latitude": lat, "Longitude": lon})
            print(f"✅ ({lat}, {lon})" if lat else "❌ Thất bại")
            time.sleep(1.2)
    os.makedirs('../data', exist_ok=True)
    with open(PROVINCES_JSON_PATH, 'w', encoding='utf-8') as f:
        json.dump(coords_data, f, ensure_ascii=False, indent=2)
    print(f"✅ Lưu vào: {PROVINCES_JSON_PATH}")

df_provinces = pd.DataFrame(coords_data)
display(df_provinces.head())

✅ provinces.json đã có đủ 63 tỉnh → Bỏ qua Geocoding.


,Code,Province,Latitude,Longitude
0,01,Hà Nội,21.0283,105.8540
1,02,Thành phố Hồ Chí Minh,10.7737,106.7166
2,03,Hải Phòng,20.8831,106.6790
3,04,Đà Nẵng,16.0685,108.2240
4,05,Hà Giang,22.8168,104.9505


<a id="debug"></a>
## 🔬 3. Kiểm tra API (Debug)
Chạy ô này trước để xem chính xác API đang trả về lỗi gì.

In [37]:
# ── Test thử 1 request nhỏ với Hà Nội, chỉ 7 ngày ──────────────────
print("🔬 Đang test API với 1 request nhỏ (Hà Nội, 7 ngày)...")

test_url    = "https://air-quality-api.open-meteo.com/v1/air-quality"
test_params = {
    "latitude":   21.0283,
    "longitude":  105.854,
    "hourly":     "pm2_5,european_aqi",   # Chỉ 2 biến để test
    "start_date": "2024-01-01",
    "end_date":   "2024-01-07",
    "timezone":   "Asia/Ho_Chi_Minh"
}

try:
    r = requests.get(test_url, params=test_params, timeout=30)
    print(f"HTTP Status Code : {r.status_code}")
    print(f"URL thực tế gọi  : {r.url}")

    if r.status_code == 200:
        data = r.json()
        print("✅ API hoạt động bình thường!")
        print(f"   Số giờ nhận được: {len(data.get('hourly', {}).get('time', []))}")
    else:
        print(f"❌ Lỗi! Response body:")
        print(r.text[:500])

except Exception as e:
    print(f"❌ Exception: {type(e).__name__}: {e}")

🔬 Đang test API với 1 request nhỏ (Hà Nội, 7 ngày)...
HTTP Status Code : 200
URL thực tế gọi  : https://air-quality-api.open-meteo.com/v1/air-quality?latitude=21.0283&longitude=105.854&hourly=pm2_5%2Ceuropean_aqi&start_date=2024-01-01&end_date=2024-01-07&timezone=Asia%2FHo_Chi_Minh
✅ API hoạt động bình thường!
   Số giờ nhận được: 168


<a id="3"></a>
## 🌤️ 4. Crawl Dữ liệu Không khí Lịch sử (Hourly)

**Kỹ thuật Chunking:** Chia nhỏ thành các chunk **6 tháng** để tránh lỗi response quá lớn.

> ⚠️ **Chỉ chạy ô này sau khi ô Debug ở trên đã xác nhận API hoạt động.**

In [38]:
def date_chunks(start_str, end_str, months=6):
    """Chia khoảng thời gian dài thành các chunk nhỏ."""
    start   = datetime.strptime(start_str, "%Y-%m-%d")
    end     = datetime.strptime(end_str,   "%Y-%m-%d")
    chunks  = []
    current = start
    while current < end:
        next_chunk = current + timedelta(days=months * 30)
        chunk_end  = min(next_chunk - timedelta(days=1), end)
        chunks.append((current.strftime("%Y-%m-%d"), chunk_end.strftime("%Y-%m-%d")))
        current = next_chunk
    return chunks


def fetch_air_quality_hourly(province, start_date, end_date, max_retries=3):
    """Lấy dữ liệu hourly theo từng chunk 6 tháng, retry tự động khi lỗi."""
    url        = "https://air-quality-api.open-meteo.com/v1/air-quality"
    all_chunks = []

    for chunk_start, chunk_end in date_chunks(start_date, end_date, months=6):
        params = {
            "latitude":   province["Latitude"],
            "longitude":  province["Longitude"],
            "hourly":     HOURLY_VARS,
            "start_date": chunk_start,
            "end_date":   chunk_end,
            "timezone":   "Asia/Ho_Chi_Minh"
        }

        for attempt in range(1, max_retries + 1):
            try:
                r = requests.get(url, params=params, timeout=60)

                # In chi tiết lỗi nếu không phải 200
                if r.status_code != 200:
                    print(f"\n     ❌ HTTP {r.status_code} [{chunk_start}→{chunk_end}]: {r.text[:200]}")
                    r.raise_for_status()

                data   = r.json()
                hourly = data.get("hourly", {})
                if not hourly or "time" not in hourly:
                    break

                df = pd.DataFrame(hourly)
                df["time"]     = pd.to_datetime(df["time"])
                df["Province"] = province["Province"]
                df["Code"]     = province["Code"]
                df["Region"]   = province.get("Region", "")
                all_chunks.append(df)
                break

            except Exception as e:
                print(f"\n     ⚠️  [{chunk_start}→{chunk_end}] Lần {attempt}/{max_retries}: {type(e).__name__}")
                if attempt < max_retries:
                    time.sleep(2 ** attempt)

        time.sleep(0.2)

    return pd.concat(all_chunks, ignore_index=True) if all_chunks else None


# Preview các chunk sẽ được tạo ra
chunks_preview = date_chunks(START_DATE, END_DATE)
print(f"✅ Mỗi tỉnh sẽ được chia thành {len(chunks_preview)} chunk:")
for c in chunks_preview:
    print(f"   ↳ {c[0]} → {c[1]}")

✅ Mỗi tỉnh sẽ được chia thành 6 chunk:
   ↳ 2024-01-01 → 2024-06-28
   ↳ 2024-06-29 → 2024-12-25
   ↳ 2024-12-26 → 2025-06-23
   ↳ 2025-06-24 → 2025-12-20
   ↳ 2025-12-21 → 2026-06-18
   ↳ 2026-06-19 → 2026-07-06


In [39]:
valid_provinces = df_provinces.dropna(subset=['Latitude', 'Longitude']).to_dict('records')

print(f"🚀 Bắt đầu crawl dữ liệu hourly...")
print(f"   Số tỉnh   : {len(valid_provinces)}")
print(f"   Khoảng    : {START_DATE} → {END_DATE}")
print("-" * 60)

all_dfs = []
failed  = []

for i, province in enumerate(valid_provinces, 1):
    name = province['Province']
    print(f"[{i:02d}/{len(valid_provinces)}] {name:<25}", end="", flush=True)
    df = fetch_air_quality_hourly(province, START_DATE, END_DATE)
    if df is not None and not df.empty:
        all_dfs.append(df)
        print(f"✅  {len(df):,} dòng")
    else:
        failed.append(name)
        print("❌  Thất bại")
    time.sleep(0.15)

print("-" * 60)
print(f"✅ Hoàn thành: {len(all_dfs)}/{len(valid_provinces)} tỉnh thành công")
if failed:
    print(f"❌ Thất bại  : {', '.join(failed)}")

🚀 Bắt đầu crawl dữ liệu hourly...
   Số tỉnh   : 63
   Khoảng    : 2024-01-01 → 2026-07-06
------------------------------------------------------------
[01/63] Hà Nội                   ✅  22,032 dòng
[02/63] Thành phố Hồ Chí Minh    ✅  22,032 dòng
[03/63] Hải Phòng                ✅  22,032 dòng
[04/63] Đà Nẵng                  ✅  22,032 dòng
[05/63] Hà Giang                 ✅  22,032 dòng
[06/63] Cao Bằng                 ✅  22,032 dòng
[07/63] Lai Châu                 ✅  22,032 dòng
[08/63] Lào Cai                  ✅  22,032 dòng
[09/63] Tuyên Quang              ✅  22,032 dòng
[10/63] Lạng Sơn                 ✅  22,032 dòng
[11/63] Bắc Kạn                  ✅  22,032 dòng
[12/63] Thái Nguyên              ✅  22,032 dòng
[13/63] Yên Bái                  ✅  22,032 dòng
[14/63] Sơn La                   ✅  22,032 dòng
[15/63] Phú Thọ                  ✅  22,032 dòng
[16/63] Vĩnh Phúc                ✅  22,032 dòng
[17/63] Quảng Ninh               ✅  22,032 dòng
[18/63] Bắc Giang               

In [40]:
df_final = pd.concat(all_dfs, ignore_index=True)

df_final.rename(columns={
    "time":             "Timestamp",
    "pm2_5":            "PM2.5 (µg/m³)",
    "pm10":             "PM10 (µg/m³)",
    "carbon_monoxide":  "CO (µg/m³)",
    "nitrogen_dioxide": "NO2 (µg/m³)",
    "sulphur_dioxide":  "SO2 (µg/m³)",
    "ozone":            "O3 (µg/m³)",
    "dust":             "Dust (µg/m³)",
    "european_aqi":     "AQI"
}, inplace=True)

col_order = ["Timestamp", "Code", "Province", "Region",
             "AQI", "PM2.5 (µg/m³)", "PM10 (µg/m³)",
             "CO (µg/m³)", "NO2 (µg/m³)", "SO2 (µg/m³)", "O3 (µg/m³)", "Dust (µg/m³)"]
df_final = df_final[[c for c in col_order if c in df_final.columns]]
df_final = df_final.sort_values(["Province", "Timestamp"]).reset_index(drop=True)

OUTPUT_PATH = "../data/vietnam_air_quality_dataset.csv"
df_final.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
print(f"📁 Đã lưu tại  : {OUTPUT_PATH}")
print(f"   Tổng số dòng: {len(df_final):,}")
display(df_final.head())

📁 Đã lưu tại  : ../data/vietnam_air_quality_dataset.csv
   Tổng số dòng: 1,388,016


,Timestamp,Code,Province,Region,AQI,PM2.5 (µg/m³),PM10 (µg/m³),CO (µg/m³),NO2 (µg/m³),SO2 (µg/m³),O3 (µg/m³),Dust (µg/m³)
0,2024-01-01 00:00:00,51,An Giang,,57,28.6,44.1,392.0,10.1,13.7,58.0,0.0
1,2024-01-01 01:00:00,51,An Giang,,56,23.9,37.5,348.0,9.6,12.8,53.0,0.0
2,2024-01-01 02:00:00,51,An Giang,,55,19.7,31.2,315.0,9.5,11.8,48.0,0.0
3,2024-01-01 03:00:00,51,An Giang,,54,16.5,26.5,291.0,9.6,10.7,42.0,0.0
4,2024-01-01 04:00:00,51,An Giang,,51,14.7,23.2,290.0,10.2,10.3,38.0,0.0


<a id="4"></a>
## 🔍 5. Kiểm thử Dữ liệu (Data Validation)

In [41]:
print("=" * 60)
print("           BÁO CÁO KIỂM THỬ DỮ LIỆU")
print("=" * 60)

n_provinces = df_final['Province'].nunique()
expected    = len(valid_provinces)
status      = "✅ PASS" if n_provinces == expected else f"❌ FAIL (thiếu {expected - n_provinces} tỉnh)"
print(f"\n[1] Số tỉnh           : {n_provinces}/{expected}  →  {status}")
print(f"    Tổng số dòng       : {len(df_final):,}")

min_time = df_final['Timestamp'].min()
max_time = df_final['Timestamp'].max()
months   = (max_time - min_time).days / 30
status   = "✅ PASS" if months >= 6 else "❌ FAIL"
print(f"\n[2] Khoảng thời gian  : {min_time.date()} → {max_time.date()} (~{months:.1f} tháng)  →  {status}")

measure_cols = [c for c in df_final.columns if c not in ['Timestamp','Code','Province','Region']]
null_pct     = df_final[measure_cols].isnull().mean() * 100
print(f"\n[3] Tỷ lệ NULL:")
for col, pct in null_pct.items():
    print(f"    {'✅' if pct < 10 else '⚠️'}  {col:<20} {pct:.2f}%")

missing = set(p['Province'] for p in valid_provinces) - set(df_final['Province'].unique())
if missing:
    print(f"\n[4] ❌ Tỉnh bị thiếu: {', '.join(sorted(missing))}")
else:
    print(f"\n[4] ✅ Đủ {n_provinces} tỉnh.")

print("\n" + "=" * 60)
display(df_final[measure_cols].describe().round(2))

           BÁO CÁO KIỂM THỬ DỮ LIỆU

[1] Số tỉnh           : 63/63  →  ✅ PASS
    Tổng số dòng       : 1,388,016

[2] Khoảng thời gian  : 2024-01-01 → 2026-07-06 (~30.6 tháng)  →  ✅ PASS

[3] Tỷ lệ NULL:
    ✅  AQI                  0.00%
    ✅  PM2.5 (µg/m³)        0.00%
    ✅  PM10 (µg/m³)         0.00%
    ✅  CO (µg/m³)           0.00%
    ✅  NO2 (µg/m³)          0.00%
    ✅  SO2 (µg/m³)          0.00%
    ✅  O3 (µg/m³)           0.00%
    ✅  Dust (µg/m³)         0.00%

[4] ✅ Đủ 63 tỉnh.



,AQI,PM2.5 (µg/m³),PM10 (µg/m³),CO (µg/m³),NO2 (µg/m³),SO2 (µg/m³),O3 (µg/m³),Dust (µg/m³)
count,1388016.00,1388016.00,1388016.00,1388016.00,1388016.00,1388016.00,1388016.00,1388016.00
mean,47.75,22.97,27.61,330.16,9.70,9.43,74.85,0.91
std,21.88,19.35,21.08,205.19,10.78,10.94,37.87,3.31
min,5.00,0.00,0.00,66.00,0.00,0.00,0.00,0.00
25%,30.00,10.60,14.00,199.00,2.60,2.20,48.00,0.00
50%,42.00,17.20,21.90,277.00,6.00,4.90,70.00,0.00
75%,64.00,28.80,34.60,402.00,12.80,12.70,95.00,1.00
max,176.00,292.40,294.30,5784.00,126.10,128.20,418.00,110.00
